# Setup and Settings

In [ ]:
from pathlib import Path
from typing import TYPE_CHECKING

import pandas as pd
from IPython.display import display  # pyright: ignore[reportUnknownVariableType]

if TYPE_CHECKING:
    from scripts.typecheck import GaussianCopulaSynthesizer, Metadata
else:
    from sdv.metadata import Metadata
    from sdv.single_table import GaussianCopulaSynthesizer

project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)

input_paths = [project_root / r"path/to/file.xlsb"]
output_dir = project_root / r"data/synthetic"
sheet_name = 0
num_rows = 200

# Load File

In [ ]:
def read_data(path: Path, sheet_name: str | int = 0) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in {".xls", ".xlsx", ".xlsm", ".xlsb", ".ods"}:
        # pandas-stubs references the unavailable optional pyxlsb workbook type.
        return pd.read_excel(path, sheet_name=sheet_name, engine="calamine")  # pyright: ignore[reportUnknownMemberType]
    raise ValueError(f"Unsupported input format: {suffix}")


def generate_synthetic(
    data: pd.DataFrame, metadata_path: Path, num_rows: int = 200
) -> pd.DataFrame:
    training_data = {"data": data.copy()}
    metadata = Metadata.detect_from_dataframes(training_data, infer_keys=None)
    pii_columns = set(metadata.get_column_names(pii=True))
    id_columns = set(metadata.get_column_names(sdtype="id"))
    protected_columns: list[str] = []
    for number, column in enumerate(data.columns, start=1):
        values = data[column].astype("string")
        has_email = values.str.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", na=False).any()
        if column in pii_columns or has_email:
            if has_email:
                metadata.update_column(
                    column,
                    sdtype="email",
                    pii=True,
                    range_is_nullable=bool(values.isna().any()),
                )
            protected_columns.append(column)
            continue

        dtype = data[column].dtype
        if column in id_columns:
            sdtype = "id"
        elif pd.api.types.is_bool_dtype(dtype):
            sdtype = "boolean"
        elif pd.api.types.is_numeric_dtype(dtype):
            sdtype = "numerical"
        elif pd.api.types.is_datetime64_any_dtype(dtype):
            sdtype = "datetime"
        else:
            sdtype = "categorical"

        if sdtype in {"categorical", "id"}:
            codes = pd.Series(pd.factorize(values)[0], index=data.index)
            training_data["data"][column] = (
                f"synthetic_{number}_" + codes.astype("string")
            ).where(values.notna())
            protected_columns.append(column)
        metadata.update_column(
            column,
            sdtype=sdtype,
            range_is_nullable=bool(values.isna().any()),
        )
    metadata.save_to_json(str(metadata_path), mode="overwrite")
    synthesizer = GaussianCopulaSynthesizer(metadata)
    synthesizer.fit(training_data)
    synthetic_data = synthesizer.sample(num_rows=num_rows)["data"]
    for column in protected_columns:
        if synthetic_data[column].dropna().isin(data[column].dropna()).any():
            raise ValueError(
                f"Anonymization reused a source value in column {column!r}."
            )
    return synthetic_data

# Generate and Save Fake Dataset

In [ ]:
input_paths = [Path(path) for path in input_paths]
output_dir = Path(output_dir)
output_paths = [output_dir / f"{path.stem}_synthetic.csv" for path in input_paths]
if len({path.resolve() for path in output_paths}) != len(output_paths):
    raise ValueError(
        "Input file names must have distinct stems to avoid overwriting outputs."
    )
if {path.resolve() for path in input_paths} & {path.resolve() for path in output_paths}:
    raise ValueError("Output paths must differ from input paths.")

output_dir.mkdir(parents=True, exist_ok=True)
results: list[pd.DataFrame] = []
for input_path, output_path in zip(input_paths, output_paths, strict=True):
    metadata_path = output_path.with_name(f"{output_path.stem}_metadata.json")
    source_df = read_data(input_path, sheet_name=sheet_name)
    synthetic_df = generate_synthetic(
        source_df, metadata_path=metadata_path, num_rows=num_rows
    )
    synthetic_df.to_csv(output_path, index=False)
    results.append(synthetic_df)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(pd.concat(results, ignore_index=True) if results else pd.DataFrame())